In [32]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import nltk
# Download NLTK data once (stopwords, wordnet for lemmatization)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

# --- Data paths (relative to notebook in model/) ---
DATA_DIR = Path("../10_categories_of_yahoo_answers_for_nlp_tasks_csv")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"



In [30]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def preprocess_df(df):
    title = df["question_title"].fillna("").astype(str)
    content = df["question_content"].fillna("").astype(str)
    question = (title + " " + content).str.strip()
    return question.apply(normalize_text)

def tokenize_for_vectorizer(text):
    text = text.lower().strip()
    tokens = re.findall(r"\b\w+\b", text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens


# Load train and test data
train_df = pd.read_csv(TRAIN_PATH, nrows=50_000)
test_df = pd.read_csv(TEST_PATH, nrows=5_000)

# Preprocess text and store in new column 
train_df["text"] = preprocess_df(train_df)
test_df["text"] = preprocess_df(test_df)

# Target: class_index
train_df["class_index"] = pd.to_numeric(train_df["class_index"], errors="coerce").astype("Int64")
test_df["class_index"] = pd.to_numeric(test_df["class_index"], errors="coerce").astype("Int64")

# Clean:Drop rows with missing class 
train_df = train_df.dropna(subset=["class_index"])
test_df = test_df.dropna(subset=["class_index"])

y_train = train_df["class_index"].astype(int).values
y_test = test_df["class_index"].astype(int).values
X_train_raw = train_df["text"].values
X_test_raw = test_df["text"].values

print("Train size:", len(y_train), "Test size:", len(y_test))
print("Class distribution (train):\n",pd.Series(y_train).value_counts().sort_index())
print("----------")
print("Sample cleaned text:")
print(X_train_raw[:3])

Train size: 50000 Test size: 5000
Class distribution (train):
 1      3106
2      5545
3      4065
4      4493
5      6541
6      3656
7     12185
8      3518
9      3282
10     3609
Name: count, dtype: int64
----------
Sample cleaned text:
<StringArray>
[                                                         'why doesn't an optical mouse work on a glass table? or even on some surfaces?',
                                                         'what is the best off-road motorcycle trail ? long-distance trail throughout ca',
 'what is trans fat? how to reduce that? i heard that tras fat is bad for the body. why is that? where can we find it in our daily food?']
Length: 3, dtype: str


In [52]:
# 
count_vectorizer = CountVectorizer(
    tokenizer=tokenize_for_vectorizer,
    lowercase=False,
    max_features=10000,
)

X_train = count_vectorizer.fit_transform(X_train_raw)
X_test = count_vectorizer.transform(X_test_raw)
print("Feature matrix shape train:", X_train.shape, "test:", X_test.shape)

# Resulting feature matrix (sparse)
#print(X_train[4])
#print("Sample tokens (first doc):", tokenize_for_vectorizer(X_train_raw[4])[:25])

Feature matrix shape train: (50000, 10000) test: (5000, 10000)


In [53]:
# Random Baseline Model
rng = np.random.default_rng(1)
classes = np.unique(y_train)
y_pred_random = rng.choice(classes, size=len(y_test))
acc_random = accuracy_score(y_test, y_pred_random)
print("Random baseline accuracy:", round(acc_random, 4))
print(classification_report(y_test, y_pred_random, zero_division=0))

Random baseline accuracy: 0.0994
              precision    recall  f1-score   support

           1       0.08      0.11      0.09       354
           2       0.11      0.09      0.10       584
           3       0.08      0.11      0.09       385
           4       0.09      0.10      0.10       434
           5       0.12      0.10      0.11       622
           6       0.05      0.08      0.07       330
           7       0.27      0.11      0.15      1202
           8       0.06      0.09      0.07       330
           9       0.06      0.09      0.07       355
          10       0.07      0.09      0.08       404

    accuracy                           0.10      5000
   macro avg       0.10      0.10      0.09      5000
weighted avg       0.13      0.10      0.11      5000



In [54]:
# Majority Vote Baseline Model
from collections import Counter
majority_class = Counter(y_train).most_common(1)[0][0]
y_pred_majority = np.full(len(y_test), majority_class)
acc_majority = accuracy_score(y_test, y_pred_majority)
print("Majority baseline accuracy:", round(acc_majority, 4))
print("Predicted class for all:", majority_class)
print(classification_report(y_test, y_pred_majority, zero_division=0))

Majority baseline accuracy: 0.2404
Predicted class for all: 7
              precision    recall  f1-score   support

           1       0.00      0.00      0.00       354
           2       0.00      0.00      0.00       584
           3       0.00      0.00      0.00       385
           4       0.00      0.00      0.00       434
           5       0.00      0.00      0.00       622
           6       0.00      0.00      0.00       330
           7       0.24      1.00      0.39      1202
           8       0.00      0.00      0.00       330
           9       0.00      0.00      0.00       355
          10       0.00      0.00      0.00       404

    accuracy                           0.24      5000
   macro avg       0.02      0.10      0.04      5000
weighted avg       0.06      0.24      0.09      5000



In [55]:
# Logistical Regression Baseline Model 
lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", random_state=1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression accuracy:", round(acc_lr, 4))
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression accuracy: 0.5862
              precision    recall  f1-score   support

           1       0.53      0.33      0.41       354
           2       0.63      0.62      0.63       584
           3       0.63      0.61      0.62       385
           4       0.43      0.34      0.38       434
           5       0.78      0.75      0.76       622
           6       0.84      0.72      0.77       330
           7       0.44      0.63      0.52      1202
           8       0.64      0.51      0.57       330
           9       0.64      0.57      0.60       355
          10       0.75      0.59      0.66       404

    accuracy                           0.59      5000
   macro avg       0.63      0.57      0.59      5000
weighted avg       0.60      0.59      0.59      5000



In [56]:
# Random Forest Baseline Model
rf = RandomForestClassifier(n_estimators=100, max_depth=100, random_state=1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest accuracy: 0.4852
              precision    recall  f1-score   support

           1       0.72      0.20      0.31       354
           2       0.70      0.26      0.38       584
           3       0.68      0.36      0.47       385
           4       0.61      0.17      0.27       434
           5       0.73      0.69      0.71       622
           6       0.88      0.51      0.65       330
           7       0.33      0.79      0.46      1202
           8       0.69      0.31      0.43       330
           9       0.57      0.57      0.57       355
          10       0.79      0.35      0.48       404

    accuracy                           0.49      5000
   macro avg       0.67      0.42      0.47      5000
weighted avg       0.62      0.49      0.48      5000

